# Tone Classification Model
# Copyright (c) 2024 Yash Firke. All rights reserved.
# Licensed under the ToneCraft Proprietary License.

### Step 1: Import Libraries

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
from transformers import AdamW, BertForSequenceClassification, BertTokenizer

### Step 2: Prepare the Dataset

In [2]:
# Load dataset
df = pd.read_csv('../datasets/tone_classification_dataset.csv')

# Map tones to numeric labels
tone_mapping = {'Professional': 0, 'Friendly': 1}
df['tone_label'] = df['Tone'].map(tone_mapping)

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize text data
def tokenize_texts(text_list, max_length=128):
    return tokenizer(
        text_list.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

# Split into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Tokenize training and validation texts
train_encodings = tokenize_texts(train_df['Sentence'])
val_encodings = tokenize_texts(val_df['Sentence'])

# Convert labels to PyTorch tensors
train_labels = torch.tensor(train_df['tone_label'].values)
val_labels = torch.tensor(val_df['tone_label'].values)

### Step 3: Load and Prepare BERT for Fine-Tuning

In [3]:
# Load BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Set up optimizer and loss function
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Create DataLoader for training data
train_dataset = TensorDataset(train_encodings['input_ids'], train_encodings['attention_mask'], train_labels)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Set model to training mode
model.train()

# Training loop
epochs = 10
for epoch in range(epochs):
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = batch

        # Zero gradients from the previous iteration
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss  # Cross entropy loss calculated by BERT

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track total loss for this epoch
        total_loss += loss.item()

    # Calculate average loss for this epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Training Loss: {avg_loss:.4f}")

print("Training complete.")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/homebrew/lib/python3.12/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10 - Training Loss: 0.6812
Epoch 2/10 - Training Loss: 0.4541
Epoch 3/10 - Training Loss: 0.3023
Epoch 4/10 - Training Loss: 0.2067
Epoch 5/10 - Training Loss: 0.1232
Epoch 6/10 - Training Loss: 0.0829
Epoch 7/10 - Training Loss: 0.0550
Epoch 8/10 - Training Loss: 0.0379
Epoch 9/10 - Training Loss: 0.0260
Epoch 10/10 - Training Loss: 0.0183
Training complete.


### Step 4: Evaluate the Model

In [4]:
# Set the model to evaluation mode
model.eval()

# Function to make predictions and calculate metrics
def evaluate_model(encodings, labels):
    # Run the model on the input encodings without computing gradients
    with torch.no_grad():
        outputs = model(**encodings)
        
    # Get the predicted class labels
    predictions = torch.argmax(outputs.logits, dim=1)
    
    # Calculate accuracy
    accuracy = accuracy_score(labels, predictions)
    
    # Calculate precision, recall, and F1-score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    
    return accuracy, precision, recall, f1

# Evaluate on the validation set
val_accuracy, val_precision, val_recall, val_f1 = evaluate_model(val_encodings, val_labels)

# Print evaluation metrics
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation F1 Score: {val_f1:.4f}")


Validation Accuracy: 1.0000
Validation Precision: 1.0000
Validation Recall: 1.0000
Validation F1 Score: 1.0000


### Step 5: Test and Deploy the Model

In [ ]:
# Save the model and tokenizer to disk
model.save_pretrained('./tone_classification_model')
tokenizer.save_pretrained('./tone_classification_model')

# To deploy the model, you can load it like this:
# Load model and tokenizer
model = BertForSequenceClassification.from_pretrained('./tone_classification_model')
tokenizer = BertTokenizer.from_pretrained('./tone_classification_model')

# If you're using the model for inference:
def predict_tone(text):
    # Tokenize the input text
    encodings = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    model.eval()  # Set model to evaluation mode
    with torch.no_grad():
        outputs = model(**encodings)
    
    # Get the predicted class label
    prediction = torch.argmax(outputs.logits, dim=1).item()
    
    # Map label back to tone
    tone_mapping_reverse = {0: 'Professional', 1: 'Friendly'}
    return tone_mapping_reverse[prediction]

# Continuous loop for user input
while True:
    input_text = input("Enter your sentence (type 'exit' to quit): ")
    
    if input_text.lower() == 'exit':
        print("Exiting the program.")
        break
    
    predicted_tone = predict_tone(input_text)
    print(f"Predicted Tone: {predicted_tone}")


Enter your sentence (type 'exit' to quit):  Whats up


Predicted Tone: Friendly


Enter your sentence (type 'exit' to quit):  Whatsup


Predicted Tone: Friendly


Enter your sentence (type 'exit' to quit):  Hi how are you


...

Predicted Tone: Friendly


Enter your sentence (type 'exit' to quit):  Thank you for your patience. We are currently working on your request and will provide an update shortly.


Predicted Tone: Professional


Enter your sentence (type 'exit' to quit):  Hey! Sooo I’ve kinda been waiting for a while now, and it's starting to feel like my message just, I dunno, disappeared? 😂 Would be amazing if you could get back to me soon tho! Much appreciated!! 🙏


Predicted Tone: Friendly
